In [2]:
# Analysis with extended filters

import pandas as pd, json, numpy as np
from collections import Counter
from collections import defaultdict
from pyrosm import OSM

custom_filter = {
    "amenity": [
        "restaurant", "cafe", "fast_food", "bar", "pub", "nightclub",
        "food_court", "ice_cream", "biergarten",
        "bank", "atm", "bureau_de_change", "post_office", "fuel",
        "car_wash", "car_rental", "charging_station",
        "pharmacy", "hospital", "clinic", "doctors", "dentist", "veterinary",
        "school", "university", "college", "kindergarten", "library",
        "place_of_worship", "townhall", "community_centre", "courthouse",
        "cinema", "theatre", "arts_centre", "casino", "events_venue",
        "marketplace",
        "bus_station", "taxi", "police", "fire_station", "social_facility",
    ],
    "shop": True,   
    "tourism": [
        "hotel", "hostel", "motel", "guest_house", "apartment", "camp_site",
        "caravan_site", "chalet", "wilderness_hut", "alpine_hut",
        "museum", "gallery", "attraction", "viewpoint", "theme_park", "zoo",
        "aquarium", "picnic_site",
    ],
    "leisure": [
        "park", "garden", "playground", "dog_park", "nature_reserve",
        "sports_centre", "fitness_centre", "swimming_pool", "stadium",
        "bowling_alley", "water_park", "golf_course", "ice_rink", "marina",
    ],
    "historic": ["castle", "fort", "monument", "memorial", "archaeological_site"],
    "healthcare": ["laboratory", "physiotherapist", "hospice", "alternative"],
    "office": ["company", "government", "estate_agent", "lawyer", "insurance",
               "telecommunication", "accountant", "architect", "ngo",
               "employment_agency", "tax_advisor"],
    "craft": ["tailor", "shoemaker", "carpenter", "electrician", "photographer"],
    "natural": ["peak", "spring", "cave_entrance"],
    "waterway": ["waterfall"],
    "public_transport": ["station"],
    "railway": ["station", "halt"],
    "aeroway": ["aerodrome", "terminal"],
}

extra = ["ruins", "religion", "denomination", "cuisine", "brand", "operator",
         "opening_hours", "name:en", "alt_name", "addr:street",
         "addr:housenumber", "addr:city", "wheelchair", "website", "phone"]


osm = OSM("../data/raw/bih.osm.pbf")
gdf = osm.get_pois(custom_filter=custom_filter,extra_attributes=extra)
print(gdf.shape[0])


44238


In [3]:
gdf.columns.to_list()

['id',
 'lon',
 'lat',
 'tags',
 'visible',
 'version',
 'timestamp',
 'changeset',
 'addr:city',
 'addr:country',
 'addr:housenumber',
 'addr:postcode',
 'addr:place',
 'addr:street',
 'email',
 'name',
 'opening_hours',
 'operator',
 'phone',
 'ref',
 'url',
 'website',
 'amenity',
 'atm',
 'bar',
 'bbq',
 'building',
 'building:levels',
 'cafe',
 'car_wash',
 'drinking_water',
 'fast_food',
 'fountain',
 'fuel',
 'gambling',
 'ice_cream',
 'internet_access',
 'landuse',
 'office',
 'parking',
 'post_office',
 'restaurant',
 'school',
 'social_facility',
 'source',
 'start_date',
 'wikipedia',
 'agrarian',
 'alcohol',
 'appliance',
 'bicycle',
 'books',
 'butcher',
 'clothes',
 'craft',
 'fireplace',
 'hairdresser',
 'organic',
 'outdoor',
 'religion',
 'second_hand',
 'shoes',
 'shop',
 'trade',
 'vacuum_cleaner',
 'water',
 'wholesale',
 'attraction',
 'camp_site',
 'guest_house',
 'information',
 'museum',
 'tourism',
 'leisure',
 'outdoor_seating',
 'playground',
 'archaeological

## 1. Deduplication

In the EDA, duplicates were found by `lat`/`lon`. Here we compare **geometry** instead, because polygons (13,248 rows) have no `lat`/`lon`, so duplicates among them were invisible. Location alone is not enough, though: the most common OSM duplicate is the same object mapped once as a point and once as a polygon, so steps 1.3–1.6 add criteria that do not require equal geometry.

Procedure:
1. **Same geometry + identical content** → keep one row. Content means `tags` + all filled columns, excluding metadata (`id`, `timestamp`, `version`, `changeset`, `visible`, `osm_type`, `match_check`). `tags` are compared as a dictionary, not as a string.
2. **Same geometry + compatible content** → merge. No shared key has a different value after converting Cyrillic to Latin and lowercasing (e.g. `Јањач` / `Janjač`). The kept row receives the attributes it was missing. Rows with the same geometry but different content (e.g. ARIA and Telecom Centar in the same building) stay separate.
3. **Point inside polygon + same name + same category** → merge into the point.
4. **Same `wikidata` id, within 1 km, not a brand id** → merge.
5. **Point within 25 m outside a polygon + same name + same category** → merge into the point.
6. **Shared phone or website + within 50 m + same category + equal or contained name** → merge.
7. **Same content within 10 m** → report only, for manual review; nothing is dropped.

Unless stated otherwise, the row with the highest `version`, then the newest `timestamp`, is kept. The ids of dropped rows are stored in `merged_ids`.

In [4]:
import geopandas as gpd
from geoir.utils import build_attrs, parse_tags, SKIP

# metadata: describes the OSM record, not the object itself, so it is not compared
META = SKIP | {"match_check"}
HELPER = {"geom_key", "content_key", "merged_ids"}   # helper columns created in this step
CONTENT_COLS = [c for c in gdf.columns if c not in META | HELPER]

def clean_attrs(row):
    """tags + filled columns, without metadata; strings stripped of surrounding whitespace."""
    d = {}
    for k, v in build_attrs(row).items():
        if k in META or k in HELPER:
            continue
        if isinstance(v, np.generic):
            v = v.item()
        if isinstance(v, str):
            v = v.strip()
            if not v:
                continue
        d[k] = v
    return d

def content_key(row):
    # sort_keys: the key order inside the tags JSON must not make a difference
    return json.dumps(clean_attrs(row), sort_keys=True, ensure_ascii=False, default=str)

# normalize(): the same polygon written from a different start vertex/direction gets the same key
gdf["geom_key"] = gdf.geometry.normalize().to_wkb()
gdf["content_key"] = gdf.apply(content_key, axis=1)
gdf["merged_ids"] = [[] for _ in range(len(gdf))]

# which row is kept in a group: most edited (version), then newest (timestamp)
gdf = gdf.sort_values(["version", "timestamp"], ascending=False)

same_geom = gdf.duplicated("geom_key", keep=False)
print(f"Rows sharing geometry: {same_geom.sum()} in {gdf.loc[same_geom, 'geom_key'].nunique()} groups")
print(gdf.loc[same_geom].geometry.geom_type.value_counts().to_string())

Rows sharing geometry: 24 in 12 groups
Point      22
Polygon     2


### 1.1 Same geometry, identical content → keep one row

In [5]:
n_before = len(gdf)
keys = ["geom_key", "content_key"]
first_id = gdf.groupby(keys, sort=False)["id"].transform("first")   # first = highest version, then newest
dropped = gdf[gdf["id"] != first_id]

exact_groups = gdf[gdf.duplicated(keys, keep=False)].assign(
    group=lambda x: x.groupby(keys, sort=False).ngroup(),
    keep=lambda x: x["id"] == first_id[x.index],
)
print(f"Identical groups: {exact_groups['group'].nunique()}, dropping {len(dropped)} rows")
display(exact_groups[["group", "id", "keep", "version", "timestamp", "osm_type",
                      "name", "amenity", "shop", "natural", "tourism"]]
        .sort_values(["group", "keep"], ascending=[True, False]))

merged = dropped.groupby(first_id[dropped.index])["id"].agg(list)
gdf = gdf[gdf["id"] == first_id].copy()
gdf["merged_ids"] = [m + merged.get(i, []) for i, m in zip(gdf["id"], gdf["merged_ids"])]
print(f"{n_before} -> {len(gdf)} rows")

Identical groups: 9, dropping 9 rows


,group,id,keep,version,timestamp,osm_type,name,amenity,shop,natural,tourism
4599,0,1945411050,True,3,1752107744,node,Veliki Cicibor,NaN,NaN,peak,NaN
27890,0,12990672696,False,1,1752107744,node,Veliki Cicibor,NaN,NaN,peak,NaN
30375,1,14062219002,True,1,1785435621,node,NaN,cafe,NaN,NaN,NaN
30374,1,14062210401,False,1,1785435188,node,NaN,cafe,NaN,NaN,NaN
30015,2,13913344849,True,1,1780703383,node,Lovreć,NaN,NaN,peak,NaN
30017,2,13913377997,False,1,1780702581,node,Lovreć,NaN,NaN,peak,NaN
29664,3,13724245387,True,1,1775946472,node,NaN,NaN,NaN,NaN,viewpoint
29662,3,13722520163,False,1,1775889295,node,NaN,NaN,NaN,NaN,viewpoint
29665,4,13724245388,True,1,1775946472,node,Voda Luka,NaN,NaN,spring,NaN
29663,4,13722520164,False,1,1775889295,node,Voda Luka,NaN,NaN,spring,NaN


44238 -> 44229 rows


### 1.2 Same geometry, compatible content → merge

Rows are merged when no shared attribute differs after normalization (Cyrillic → Latin, lowercase, surrounding whitespace removed). An attribute present in only one row does not count as a difference. Typos, missing diacritics (`č` vs `c`), quotes and abbreviations do count as differences, so such rows stay separate. The remaining groups are kept apart, and the attributes they differ in are printed.

In [6]:
CYR = dict(zip("абвгдђежзијклљмнњопрстћуфхцчџш",
               ["a", "b", "v", "g", "d", "đ", "e", "ž", "z", "i", "j", "k", "l", "lj", "m",
                "n", "nj", "o", "p", "r", "s", "t", "ć", "u", "f", "h", "c", "č", "dž", "š"]))

def norm_val(v):
    # Cyrillic -> Latin + lowercase, so that 'Јањач' and 'Janjač' are the same value
    if isinstance(v, str):
        return "".join(CYR.get(ch, ch) for ch in v.strip().casefold())
    return v

def compatible(a, b):
    # no shared key has a different value
    return all(norm_val(a[k]) == norm_val(b[k]) for k in a.keys() & b.keys())

def merge_rows(keep, drops, reason, verbose=True):
    """Merge rows `drops` into row `keep` (index labels): fill missing columns and tags,
    record the dropped ids in merged_ids, then remove the dropped rows from gdf."""
    global gdf
    for d in drops:
        if verbose:
            print(f"[{reason}] merging id {gdf.at[d, 'id']} ({gdf.at[d, 'name']}) "
                  f"into id {gdf.at[keep, 'id']} ({gdf.at[keep, 'name']})")
        # fill empty columns of the kept row from the dropped row
        for c in CONTENT_COLS:
            if pd.isna(gdf.at[keep, c]) and not pd.isna(gdf.at[d, c]):
                gdf.at[keep, c] = gdf.at[d, c]
        # merge tags; on a shared key the kept row's value wins
        tags = {**parse_tags(gdf.at[d, "tags"]), **parse_tags(gdf.at[keep, "tags"])}
        gdf.at[keep, "tags"] = json.dumps(tags, ensure_ascii=False) if tags else None
        gdf.at[keep, "merged_ids"] = gdf.at[keep, "merged_ids"] + [gdf.at[d, "id"]] + gdf.at[d, "merged_ids"]
    gdf = gdf.drop(index=drops)

same_geom = gdf.duplicated("geom_key", keep=False)
merges, kept_apart = [], []
for _, grp in gdf[same_geom].groupby("geom_key", sort=False):   # gdf is already sorted by version, timestamp
    clusters = []
    for idx, row in grp.iterrows():
        a = clean_attrs(row)
        for cl in clusters:
            if compatible(cl["attrs"], a):
                cl["attrs"] = {**a, **cl["attrs"]}   # values of the kept row take precedence
                cl["drop"].append(idx)
                break
        else:
            clusters.append({"keep": idx, "attrs": a, "drop": []})
    merges += [cl for cl in clusters if cl["drop"]]
    if len(clusters) > 1:
        kept_apart.append(grp)

for cl in merges:
    merge_rows(cl["keep"], cl["drop"], "same geometry")

print(f"\nSame geometry, different content (kept separate): {len(kept_apart)} groups")
for grp in kept_apart:
    a, b = (clean_attrs(r) for _, r in grp.head(2).iterrows())
    diff = sorted(k for k in a.keys() & b.keys() if norm_val(a[k]) != norm_val(b[k]))
    print(f"  id {grp['id'].tolist()}  name {grp['name'].tolist()}  differ in: {diff}")
print(f"\nTotal rows: {len(gdf)}")

[same geometry] merging id 1983711902 (Janjač) into id 1644665074 (Јањач)

Same geometry, different content (kept separate): 2 groups
  id [7909276377, 14288818672]  name ['ARIA', 'Telecom Centar Sarajevo City']  differ in: ['name', 'opening_hours', 'shop', 'website']
  id [3567237597, 2024884165]  name [nan, nan]  differ in: ['shop']

Total rows: 44228


### 1.3 Point inside polygon, same name and category → merge

The most common OSM duplicate is one object mapped twice: as a point (node) and as an area (building or site outline). Their geometries are never equal, so step 1.1 cannot find them.

A point and a polygon are merged when the point lies inside the polygon **and** both have the same normalized name **and** the same primary category. Both conditions are needed:
- the same name alone is not enough, e.g. the *Bingo* supermarket inside the *Bingo* mall, or the *Vrelo Bosne* spring inside the *Vrelo Bosne* park;
- the same category alone is not enough, e.g. two unnamed cafés in the same building.

The **point** is kept because it has the precise location and usually more attributes; the polygon fills in what is missing. A polygon that contains several matching points is ambiguous, and pairs with the same name but a different category are only listed in `pip_review` for later review.

In [7]:
CAT_KEYS = ["amenity", "shop", "tourism", "leisure", "historic", "healthcare", "office",
            "craft", "natural", "waterway", "railway", "public_transport", "aeroway"]

def text_key(s):
    """Normalized text for comparison; missing values become '' so they never match."""
    return s.map(lambda v: norm_val(v) if isinstance(v, str) else "")

def primary_category(df):
    # value of the first filled category key, in priority order; "yes" says nothing about the type
    return df[CAT_KEYS].replace("yes", np.nan).bfill(axis=1).iloc[:, 0]

gtype = gdf.geometry.geom_type
pairs = (gpd.sjoin(gdf.loc[gtype == "Point", ["geometry"]],
                   gdf.loc[gtype.isin(["Polygon", "MultiPolygon"]), ["geometry"]], predicate="within")
         .reset_index(names="pt").rename(columns={"index_right": "poly"})[["pt", "poly"]])

name_k, cat_k = text_key(gdf["name"]), text_key(primary_category(gdf))
for side in ("pt", "poly"):
    pairs[f"id_{side}"] = gdf.loc[pairs[side], "id"].to_numpy()
    pairs[f"name_{side}"] = gdf.loc[pairs[side], "name"].to_numpy()
    pairs[f"cat_{side}"] = cat_k.loc[pairs[side]].to_numpy()
same_name = (name_k.loc[pairs["pt"]].to_numpy() == name_k.loc[pairs["poly"]].to_numpy()) & (pairs["name_pt"].notna().to_numpy())
same_cat = (pairs["cat_pt"] == pairs["cat_poly"]) & (pairs["cat_pt"] != "")

match = pairs[same_name & same_cat]
ambiguous = match.duplicated("poly", keep=False)   # several matching points in one polygon -> unclear which one
to_merge = match[~ambiguous]
for pt, grp in to_merge.groupby("pt"):
    merge_rows(pt, grp["poly"].tolist(), "point in polygon", verbose=False)   # keep the point

cols = ["id_pt", "id_poly", "name_pt", "name_poly", "cat_pt", "cat_poly"]
pip_review = pd.concat([match[ambiguous][cols].assign(reason="several points in polygon"),
                        pairs[same_name & ~same_cat][cols].assign(reason="same name, different category")])
print(f"Point-in-polygon pairs: {len(pairs)}")
print(f"  same name + same category: {len(match)}  -> merged: {len(to_merge)}, ambiguous: {ambiguous.sum()}")
print(f"  same name, different category (review): {(same_name & ~same_cat).sum()}")
print(f"Total rows: {len(gdf)}")
display(to_merge[cols].head(20))
pip_review

Point-in-polygon pairs: 4604
  same name + same category: 58  -> merged: 51, ambiguous: 7
  same name, different category (review): 57
Total rows: 44177


,id_pt,id_poly,name_pt,name_poly,cat_pt,cat_poly
74,559952410,111253400,Muzej Republike Srpske,Muzej Republike Srpske,museum,museum
97,524787616,922195660,Pošte Srpske,Pošte Srpske,post_office,post_office
307,2190833609,1175570755,"Pošte Srpske, 78250 Laktaši","Pošte Srpske, 78250 Laktaši",post_office,post_office
334,2030224274,221059780,Bingo,Bingo,supermarket,supermarket
389,5314988499,1422601505,Dying Lion,Dying Lion,memorial,memorial
393,3802703579,849501760,Nešković,Nešković,fuel,fuel
398,7212593698,772536712,Apartman GM,Apartman GM,apartment,apartment
439,2190831109,480676318,Nestro,Nestro,fuel,fuel
480,1708601264,756238405,Ambulanta Tilava,Ambulanta Tilava,clinic,clinic
493,2064604725,202808757,"Triglav, Tehnički pregled vozila","Triglav, Tehnički pregled vozila",car_repair,car_repair


,id_pt,id_poly,name_pt,name_poly,cat_pt,cat_poly,reason
1867,10893658905,392873896,AC Boras,AC Boras,car,car,several points in polygon
1869,10893658907,392873896,AC Boras,AC Boras,car,car,several points in polygon
1872,9970975406,394732932,Guma M,Guma M,car,car,several points in polygon
1873,9970975407,394732932,Guma M,Guma M,car,car,several points in polygon
1876,11091945666,394732932,Guma M,Guma M,car,car,several points in polygon
...,...,...,...,...,...,...,...
4156,7400960054,168183264,Prodex,Prodex,cafe,mall,"same name, different category"
4214,6163473741,658134217,Nostalgija,Nostalgija,cafe,biergarten,"same name, different category"
4260,5736681996,8425673,Morića Han,Morića Han,restaurant,mall,"same name, different category"
4496,2041086286,172385980,Flora,Flora,restaurant,cafe,"same name, different category"


### 1.4 Shared `wikidata` id → merge

`wikidata` (e.g. `Q3490665`) identifies one real-world object, so two rows with the same id describe the same thing even when their names differ (*Vječna vatra* / *Vječna Vatra memorial*). There are two exceptions:
- **Brands:** chains such as Telemach carry the brand's id on every store. Ids that also appear as `brand:wikidata` are skipped.
- **Mistagging:** the same id on objects kilometers apart (e.g. a bike service and a bike park). Rows are merged only when they are within 1 km of each other. The distance is measured between geometries, so a point inside a large polygon has distance 0.

A point is kept over a polygon. Groups that are not merged appear in the table as `kept separate`.

In [8]:
import itertools

WD_MAX_M = 1000   # rows sharing a wikidata id further apart than this are left alone

tags_dict = gdf["tags"].map(parse_tags)
wikidata = tags_dict.map(lambda d: d.get("wikidata"))
# a wikidata id that is also used as brand:wikidata identifies a chain, not one object
brand_ids = set(tags_dict.map(lambda d: d.get("brand:wikidata")).dropna())
wikidata = wikidata[wikidata.notna() & ~wikidata.isin(brand_ids)]
shared = wikidata[wikidata.duplicated(keep=False)]

utm = gdf.geometry.to_crs(32634)   # UTM 34N, meters
is_point = gdf.geometry.geom_type == "Point"
report = []
for qid, idx in shared.groupby(shared).groups.items():
    # geometry distance: 0 when a point lies inside the polygon, unlike the distance between centroids
    max_d = max(a.distance(b) for a, b in itertools.combinations(utm.loc[idx], 2))
    decision = "merged" if max_d <= WD_MAX_M else "kept separate"
    report.append({"wikidata": qid, "ids": gdf.loc[idx, "id"].tolist(), "names": gdf.loc[idx, "name"].tolist(),
                   "osm_type": gdf.loc[idx, "osm_type"].tolist(), "max_dist_m": round(max_d), "decision": decision})
    if decision == "merged":
        # keep a point if there is one (gdf is already sorted by version, timestamp)
        keep = next((i for i in idx if is_point[i]), idx[0])
        merge_rows(keep, [i for i in idx if i != keep], "wikidata", verbose=False)

print(f"Wikidata ids excluded as brands: {len(brand_ids)}")
print(f"Groups sharing a wikidata id: {len(report)}")
print(f"Total rows: {len(gdf)}")
pd.DataFrame(report).sort_values("max_dist_m")

Wikidata ids excluded as brands: 190
Groups sharing a wikidata id: 9
Total rows: 44169


,wikidata,ids,names,osm_type,max_dist_m,decision
0,Q12981899,"[126210754, 1399701603]","[Štrbački buk, Štrbački buk]","[way, node]",0,merged
7,Q64066244,"[692745607, 5568657138]","[Park Ravne 2, Park “Ravne 2”]","[way, node]",0,merged
5,Q43951249,"[1208624885, 11041800145]","[Zelena pećina, Zelena pećina]","[way, node]",0,merged
4,Q3490665,"[1588411957, 19467025]","[Vječna vatra, Vječna Vatra memorial]","[node, relation]",0,merged
3,Q2535058,"[2625909495, 16338534]","[Vrelo Bune, Buna Wellspring, protected spring...","[node, relation]",3,merged
1,Q137646857,"[20030892, 10933440915]","[Biciklistički park Evolution, Biciklistički p...","[relation, node]",17,merged
2,Q192050,"[5300914876, 11096822459]","[nan, The monument to the victims of the Saraj...","[node, node]",18,merged
6,Q60751261,"[11924948877, 11924948878]","[Floded Trebišnjica Spring, Floded Trebišnjica...","[node, node]",576,merged
8,Q7696320,"[7190961667, 10223345464, 10223345463, 1022334...","[Telemach, Telemach, Telemach, Telemach]","[node, node, node, node]",63554,kept separate


### 1.5 Point near polygon, same name and category → merge

An extension of 1.3. A point is often placed at the entrance or on the street, just **outside** its building outline (*Radon Plaza Hotel* 15 m, *Pijaca Grbavica* 25 m), so the inside test misses it.

When a point lies outside a polygon but within 25 m of it, and both have the same normalized name and the same primary category, the polygon is merged into the point. The same-name rule makes this safe: of about 6,800 point–polygon pairs this close, only 10 match, and all are real duplicates. A point or polygon involved in more than one match is marked ambiguous and not merged.

In [9]:
NEAR_POLY_M = 25   # how far outside its polygon a point may lie

utm = gdf.geometry.to_crs(32634)   # UTM 34N, meters
gtype = gdf.geometry.geom_type
polys_buf = gpd.GeoDataFrame(geometry=utm[gtype.isin(["Polygon", "MultiPolygon"])].buffer(NEAR_POLY_M))
near = (gpd.sjoin(gpd.GeoDataFrame(geometry=utm[gtype == "Point"]), polys_buf, predicate="within")
        .reset_index(names="pt").rename(columns={"index_right": "poly"})[["pt", "poly"]])
near["dist_m"] = [utm[p].distance(utm[q]) for p, q in zip(near["pt"], near["poly"])]
# points inside the polygon were handled in 1.3
near = near[~np.array([utm[p].within(utm[q]) for p, q in zip(near["pt"], near["poly"])], dtype=bool)]

name_k, cat_k = text_key(gdf["name"]), text_key(primary_category(gdf))
same_name = (name_k.loc[near["pt"]].to_numpy() == name_k.loc[near["poly"]].to_numpy()) & (name_k.loc[near["pt"]].to_numpy() != "")
same_cat = (cat_k.loc[near["pt"]].to_numpy() == cat_k.loc[near["poly"]].to_numpy()) & (cat_k.loc[near["pt"]].to_numpy() != "")
match = near[same_name & same_cat].copy()
ambiguous = match.duplicated("poly", keep=False) | match.duplicated("pt", keep=False)
match["id_pt"] = gdf.loc[match["pt"], "id"].to_numpy()
match["id_poly"] = gdf.loc[match["poly"], "id"].to_numpy()
match["name"] = gdf.loc[match["pt"], "name"].to_numpy()
match["category"] = cat_k.loc[match["pt"]].to_numpy()
match["decision"] = np.where(ambiguous, "ambiguous (review)", "merged")

for pt, poly in zip(match.loc[~ambiguous, "pt"], match.loc[~ambiguous, "poly"]):
    merge_rows(pt, [poly], "point near polygon", verbose=False)   # keep the point

print(f"Points outside a polygon but within {NEAR_POLY_M} m of it: {len(near)}")
print(f"  same name + same category: {len(match)}  -> merged: {(~ambiguous).sum()}, ambiguous: {ambiguous.sum()}")
print(f"Total rows: {len(gdf)}")
match[["id_pt", "id_poly", "name", "category", "dist_m", "decision"]].sort_values("dist_m").round(1)

Points outside a polygon but within 25 m of it: 6798
  same name + same category: 10  -> merged: 10, ambiguous: 0
Total rows: 44159


,id_pt,id_poly,name,category,dist_m,decision
10722,3216262118,315534737,STUDIO 1,restaurant,0.0,merged
6209,4054699588,223241764,Studio Borak,photographer,1.3,merged
450,1300694703,200070310,Vatrogasno društvo Tarčin,fire_station,2.0,merged
1013,5626862829,1284682025,Džamija Slatina,place_of_worship,3.9,merged
10214,5533775695,577183352,ToGo,cafe,4.8,merged
5198,5148545010,530116004,Općinski sud u Sarajevu,courthouse,5.5,merged
1056,7438334990,17632419,Tvrđava Kruševac,archaeological_site,7.5,merged
5012,8976666737,970038819,GEA,cafe,8.8,merged
112,673355097,850107288,Radon Plaza Hotel,hotel,14.9,merged
2159,2005529316,191046576,Pijaca Grbavica,marketplace,24.6,merged


### 1.6 Shared phone or website + strict name rule → merge

On its own, a shared phone number or website is **not** a duplicate signal:
- far apart, it marks a chain (Petrol, UniCredit, Gazprom; one phone number is shared by 123 rows);
- close together, it usually marks *related* objects: a hotel and its restaurant (*Ibis Styles* / *Igman Rooftop Bistro*), a mall and its shops (*Šipad Doboj*), a bank and its ATM, two police units in one building.

It works only together with strict extra conditions. Two rows are merged when **all** of these hold:
- they share a normalized phone number or website (a value shared by more than 30 rows is ignored as a hotline);
- they are within 50 m of each other;
- they have the same primary category;
- the names are equal, or one contains the other, after normalization (*Toplik* / *Restoran Toplik*);
- the names contain the same numbers, so *Gazprom Tuzla 1* / *Gazprom Tuzla 2* and *Moja apoteka Boska* / *Moja apoteka Boska 2* stay separate.

A point is kept over a polygon.

In [10]:
import re

CONTACT_MAX_M = 50

def norm_phone(p):
    # digits only, without the country code and leading zeros: '+387 33 123-456' -> '33123456'
    d = re.sub(r"\D", "", p)
    d = re.sub(r"^(00387|387)", "", d).lstrip("0")
    return d if len(d) >= 7 else None

def norm_web(w):
    # 'https://www.Example.ba/' -> 'example.ba'
    w = re.sub(r"^(https?://)?(www\.)?", "", w.strip().lower())
    return w.rstrip("/") or None

def contact_values(i):
    """All normalized phone numbers and websites of row i (columns + tags)."""
    tags = parse_tags(gdf.at[i, "tags"])
    out = set()
    for key in ["phone", "contact:phone", "contact:mobile", "mobile"]:
        v = gdf.at[i, key] if key in gdf.columns else tags.get(key)
        if isinstance(v, str):
            out |= {("phone", n) for part in re.split(r"[;,/]", v) if (n := norm_phone(part))}
    for key in ["website", "contact:website", "url"]:
        v = gdf.at[i, key] if key in gdf.columns else tags.get(key)
        if isinstance(v, str) and (n := norm_web(v)):
            out.add(("website", n))
    return out

def similar_names(a, b):
    # equal or one contained in the other, and the same numbers ('Gazprom Tuzla 1' != 'Gazprom Tuzla 2')
    if not a or not b or not (a in b or b in a):
        return False
    return re.findall(r"\d+", a) == re.findall(r"\d+", b)

contacts = pd.DataFrame([(i, kind, val) for i in gdf.index for kind, val in contact_values(i)],
                        columns=["idx", "kind", "value"])
shared = contacts[contacts.duplicated("value", keep=False)]
print(f"Rows with phone or website: {contacts['idx'].nunique()}, values shared by several rows: {shared['value'].nunique()}")

utm = gdf.geometry.to_crs(32634)
name_k, cat_k = text_key(gdf["name"]), text_key(primary_category(gdf))
cand = []
for value, idx in shared.groupby("value")["idx"].apply(list).items():
    if len(idx) > 30:   # central hotline of a chain, not a duplicate signal
        continue
    for a, b in itertools.combinations(sorted(set(idx)), 2):
        d = utm[a].distance(utm[b])
        if d <= CONTACT_MAX_M and cat_k[a] == cat_k[b] != "" and similar_names(name_k[a], name_k[b]):
            cand.append((a, b, value, d))
cand = pd.DataFrame(cand, columns=["a", "b", "value", "dist_m"]).drop_duplicates(["a", "b"])
cand = cand.assign(id_a=gdf.loc[cand["a"], "id"].to_numpy(), id_b=gdf.loc[cand["b"], "id"].to_numpy(),
                   name_a=gdf.loc[cand["a"], "name"].to_numpy(), name_b=gdf.loc[cand["b"], "name"].to_numpy(),
                   category=cat_k.loc[cand["a"]].to_numpy())

is_point = gdf.geometry.geom_type == "Point"
n_merged = 0
for a, b in zip(cand["a"], cand["b"]):
    if a not in gdf.index or b not in gdf.index:   # already merged through another pair
        continue
    n_merged += 1
    # keep a point over a polygon, otherwise the higher version / newer row (gdf order)
    keep, drop = (a, b) if (is_point[a], -gdf.index.get_loc(a)) >= (is_point[b], -gdf.index.get_loc(b)) else (b, a)
    merge_rows(keep, [drop], "shared contact", verbose=False)

print(f"Candidate pairs: {len(cand)}, merged: {n_merged}")
print(f"Total rows: {len(gdf)}")
cand[["id_a", "id_b", "name_a", "name_b", "category", "value", "dist_m"]].sort_values("dist_m").round(1)

Rows with phone or website: 4985, values shared by several rows: 347
Candidate pairs: 10, merged: 10
Total rows: 44149


,id_a,id_b,name_a,name_b,category,value,dist_m
1,637550199,1468588079,JU Zavod za bolesti ovisnosti Kantona Sarajevo,JU Zavod za bolesti ovisnosti Kantona Sarajevo,hospital,33257750,0.0
2,194393960,680086049,Petrol,Petrol,fuel,33780091,0.0
3,6657345685,756194717,Toplik,Restoran Toplik,restaurant,57321455,0.0
11,8205375395,189256831,Cafe Bar Big Ben,Big Ben,cafe,instagram.com/caffebigben,0.0
6,10536927409,10536931041,Vulkanizer Maslo,Vulkanizer Maslo,car_repair,63284498,1.6
0,2142823971,7093893680,Pirpa,Pirpa,fast_food,33208183,15.4
4,13956950795,887531500,Heart of the nature by Armin,Armin,apartment,62114666,15.4
7,11895606009,11895606010,Don Sergio Pasticceria,Don Sergio,restaurant,66583839,15.4
5,4845221621,4944146823,Apartment Panorama Mostar,Apartment Panorama Mostar,guest_house,62833952,23.6
12,3676419925,3676419926,Kulin Dvor - ljetna bašta,Kulin Dvor,restaurant,kulindvor.com,44.0


### 1.7 Near-duplicates (review only)

Identical content with slightly shifted coordinates is not caught by comparing geometry. These pairs are **not dropped automatically**. For example, *Plivski Mlinčići* is a row of separate watermills and *Vodopad Provalije* is several waterfalls. Some of them are real duplicates, though (e.g. *Bijeli vrh* at 0.4 m, *UniCredit Bank* at 1.4 m). So `near_dups` should be reviewed manually later; a shared phone number or website (see 1.6) is a useful hint during that review.

In [11]:
# same content, named, within 10 m (but not at the same point) -> review only
NEAR_M = 10
gdf["content_key"] = gdf.apply(content_key, axis=1)   # refresh after merging
cand = gdf[gdf["name"].notna() & gdf.duplicated("content_key", keep=False)]
p = gpd.GeoDataFrame(cand[["id", "name", "content_key"]],
                     geometry=cand.geometry.representative_point(), crs=gdf.crs).to_crs(32634)  # UTM 34N, meters
pairs = gpd.sjoin(p, p.set_geometry(p.buffer(NEAR_M)), predicate="within")
pairs = pairs[(pairs["id_left"] < pairs["id_right"]) & (pairs["content_key_left"] == pairs["content_key_right"])]
pairs["dist_m"] = [p.geometry.loc[i].distance(p.geometry.loc[j]) for i, j in zip(pairs.index, pairs["index_right"])]
print(f"Pairs with identical content within {NEAR_M} m: {len(pairs)}")
near_dups = pairs[["id_left", "id_right", "name_left", "dist_m"]].sort_values("dist_m").round(1)
near_dups

Pairs with identical content within 10 m: 56


,id_left,id_right,name_left,dist_m
4901,2012695486,2325434489,Bijeli vrh,0.4
14696,7083756429,7083756430,UniCredit Bank,1.4
22135,11043594093,11525595171,MF banka,1.4
22573,11121195936,11121231235,Popov Izvor,2.2
12874,5980981955,11067755352,Vodopad Provalije,2.2
43248,1436543109,1436543114,Plivski Mlinčići,2.9
43244,1436543105,1436543106,Plivski Mlinčići,2.9
43253,1436543114,1436543115,Plivski Mlinčići,2.9
43245,1436543106,1436543107,Plivski Mlinčići,3.0
43243,1436543104,1436543108,Plivski Mlinčići,3.2


## 2. Schema Standardization & NLP Text Preparation

This section turns the deduplicated `gdf` into `gdf_clean`: one row per POI with clean structured columns and an `nlp_text` string for retrieval. The rules come from the `tags` analysis:

- **Coordinates:** every row gets `lat`/`lon`; polygons and lines get a point that lies inside them.
- **Names:** all name variants are collected, repaired (homoglyphs, `Ð`, quotes), transliterated to Latin and deduplicated into `primary_name` + `aliases`. Abbreviations (`OŠ`, `JU`, `JZU`, `MZ`, `JP`/`JKP`/`KJKP`, `TR`/`STR`, `SZR`) are expanded into extra aliases.
- **`nlp_text`** uses an allowlist: names, bilingual category label, brand/operator, descriptive tags, verbalized features, address and free text. Technical/QA keys (`source`, `check_date`, `fixme`, `wikidata`, `members`, `ref`, `level`, …) and contact/opening-hours keys never enter it.
- **Negative values (`no`)** are not written into `nlp_text` (embedding models handle negation poorly); they are kept in the structured `filters` column.
- **Structured columns:** `contact` (normalized phones in E.164, website, email, socials) and `filters` (normalized yes/no/enum features) are JSON strings; `opening_hours`, `address`, `city`, `postcode`, `wikidata` and `loc_approx` are plain columns.

### 2.1 Fill missing coordinates

Polygons, multipolygons and lines have no `lat`/`lon`. They get `geometry.representative_point()` (in EPSG:4326), which always lies inside the shape, unlike the centroid of an L- or U-shaped building.

In [12]:
assert gdf.crs.to_epsg() == 4326, gdf.crs

missing = gdf["lat"].isna() | gdf["lon"].isna()
# representative_point() always lies inside the polygon (a centroid can fall outside an L- or U-shaped one)
rp = gdf.loc[missing].geometry.representative_point()
gdf.loc[missing, "lon"] = rp.x.to_numpy()
gdf.loc[missing, "lat"] = rp.y.to_numpy()

print(f"Filled lat/lon for {missing.sum()} rows:")
print(gdf.loc[missing].geometry.geom_type.value_counts().to_string())
print(f"Rows still missing coordinates: {(gdf['lat'].isna() | gdf['lon'].isna()).sum()}")

Filled lat/lon for 13175 rows:
Polygon         13116
MultiPolygon       40
LineString         19
Rows still missing coordinates: 0


### 2.2 Attribute dictionary per row

`tags` and the filled columns are merged into one dict per row, without metadata (same as `clean_attrs` in step 1). Value fixes from the analysis: typos (`yese`, `y` → `yes`) and empty values (`unknown`, `undefined`, `none`) are dropped. Name keys are left untouched.

In [13]:
import unicodedata

VALUE_FIX = {"yese": "yes", "y": "yes"}              # typos found in the analysis
DROP_VALUES = {"unknown", "undefined", "none"}        # values that carry no information
NAME_LIKE = re.compile(r"^(name|alt_name|official_name|short_name|loc_name|int_name|old_name)(:|$)")

def row_attrs(row):
    """All attributes of a row (tags + columns, no metadata), with the value fixes applied."""
    d = {}
    for k, v in clean_attrs(row).items():
        if isinstance(v, str):
            v = re.sub(r"\s+", " ", unicodedata.normalize("NFC", v)).strip()
            if not NAME_LIKE.match(k):
                v = VALUE_FIX.get(v.lower(), v)
                if v.lower() in DROP_VALUES:
                    continue
        d[k] = v
    return d

attrs = gdf.apply(row_attrs, axis=1)
print(f"Attribute dicts built for {len(attrs)} rows, {attrs.map(len).sum()} key/value pairs")

Attribute dicts built for 44149 rows, 185676 key/value pairs


### 2.3 Names: repair, transliterate, deduplicate

For each row:
1. Collect `name`, `name:bs/hr/sr-Latn/sr/bs-Cyrl/en`, `alt_name` (+ language variants), `official_name`, `short_name`, `loc_name`, `int_name`, `old_name`, split on `;` and ` / `.
2. Repair every string: NFC; Cyrillic look-alike letters inside Latin words and vice versa (*аpoteka* → *apoteka*); Icelandic `Ð` → `Đ`; unified quotes; single spaces.
3. Transliterate to Latin and deduplicate by a comparison key (no legal form, no quotes, casefolded, no diacritics). Priority: Latin `name` → `name:bs` → `name:hr` → `name:sr-Latn` → Cyrillic `name`/`name:sr` → `name:en` → the rest.
4. The first survivor is `primary_name`; the rest are `aliases`. Abbreviation expansions are added as aliases.

The display form keeps quotes, legal forms and ALL CAPS; the `text_form` used in `nlp_text` drops legal forms and quotes and lowercases ALL CAPS names.

In [14]:
# --- script handling --------------------------------------------------------
# case-aware Cyrillic -> Latin, extending CYR from 1.2 with a few Russian/Macedonian letters found in names
CYR2LAT = dict(CYR)
CYR2LAT.update({"й": "j", "ы": "y", "э": "e", "ю": "ju", "я": "ja", "щ": "šč", "ъ": "", "ь": "", "ё": "jo",
                "і": "i", "ї": "ji", "є": "je", "ѓ": "đ", "ќ": "ć", "ѕ": "dz"})
CYR2LAT.update({c.upper(): l[:1].upper() + l[1:] for c, l in list(CYR2LAT.items()) if l})

def translit(s):
    return "".join(CYR2LAT.get(ch, ch) for ch in s)

def is_cyr(ch):
    return "Ѐ" <= ch <= "ӿ"

def is_lat(ch):
    return ch.isalpha() and "LATIN" in unicodedata.name(ch, "")

# letters that look identical in both scripts
CYR_LOOKALIKE = dict(zip("аеорсухкмтјіѕАВЕКМНОРСТХЈІЅ", "aeopcyxkmtjisABEKMHOPCTXJIS"))
LAT_LOOKALIKE = {l: c for c, l in CYR_LOOKALIKE.items() if c not in "уіѕІЅ"}

def _fix_word(m):
    w = m.group(0)
    n_cyr, n_lat = sum(map(is_cyr, w)), sum(map(is_lat, w))
    if not (n_cyr and n_lat):
        return w
    if n_lat >= n_cyr:   # Latin word with Cyrillic look-alikes: 'аpoteka' -> 'apoteka'
        return "".join(CYR_LOOKALIKE.get(ch, ch) for ch in w)
    return "".join(LAT_LOOKALIKE.get(ch, ch) for ch in w)   # Cyrillic word with Latin look-alikes

QUOTES = str.maketrans({"„": '"', "“": '"', "”": '"', "«": '"', "»": '"', "″": '"', "‘": "'", "’": "'", "`": "'"})

def tidy(s):
    """NFC, homoglyph repair, 'Ð' -> 'Đ', unified quotes, single spaces."""
    s = unicodedata.normalize("NFC", s).replace("Ð", "Đ").replace("ð", "đ")
    s = re.sub(r"\w+", _fix_word, s).translate(QUOTES).replace("''", '"')
    return re.sub(r"\s+", " ", s).strip(" ;,")

# --- text forms -------------------------------------------------------------
LEGAL_RE = re.compile(r"(?<!\w)(d\.\s?o\.\s?o\.?|doo|d\.\s?d\.|a\.\s?d\.|s\.\s?p\.)(?!\w)", re.I)

def is_all_caps(s):
    return sum(ch.isalpha() for ch in s) >= 4 and s == s.upper() and s != s.lower()

def text_form(s):
    """Version used in nlp_text: Latin script, no legal form, no quotes, ALL CAPS lowered."""
    t = translit(tidy(s))
    t = re.sub(r"\s+", " ", LEGAL_RE.sub("", t).replace('"', "")).strip(" ,.-;")
    if is_all_caps(t):
        t = t.lower()
    return t or translit(tidy(s))

def match_key(s):
    """Comparison key: text form, casefolded, without diacritics and punctuation."""
    t = text_form(s).casefold().replace("đ", "dj")
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", " ", t).strip()

# --- abbreviations ----------------------------------------------------------
ABBREVIATIONS = [   # longest first; matched case-sensitively as whole tokens
    (r"K\.?J\.?K\.?P\.?", "kantonalno javno komunalno preduzeće"),
    (r"J\.?K\.?P\.?", "javno komunalno preduzeće"),
    (r"J\.?Z\.?U\.?", "javna zdravstvena ustanova"),
    (r"S\.?Z\.?R\.?", "samostalna zanatska radnja"),
    (r"S\.?T\.?R\.?", "samostalna trgovinska radnja"),
    (r"O\.?\s?[Šš]\.?", "osnovna škola"),
    (r"J\.?\s?U\.?", "javna ustanova"),
    (r"J\.?\s?P\.?", "javno preduzeće"),
    (r"M\.?\s?Z\.?", "mjesna zajednica"),
    (r"T\.?\s?R\.?", "trgovinska radnja"),
]
ABBREVIATIONS = [(re.compile(rf"(?<![\w.]){p}(?!\w)"), full) for p, full in ABBREVIATIONS]

def expand_abbr(s):
    """'OŠ Hasan Kikić' -> 'osnovna škola Hasan Kikić'; None when there is nothing to expand."""
    t = translit(tidy(s))
    new = t
    for rx, full in ABBREVIATIONS:
        new = rx.sub(full, new)
    if new == t:
        return None
    return text_form(new.lower() if is_all_caps(t) else new)

# --- name variants ----------------------------------------------------------
NAME_KEYS = ["name", "name:bs", "name:hr", "name:sr-Latn", "name:sr", "name:bs-Cyrl", "name:en",
             "alt_name", "alt_name:bs", "alt_name:hr", "alt_name:sr-Latn", "alt_name:sr", "alt_name:en",
             "official_name", "short_name", "loc_name", "int_name", "old_name"]

def build_names(a):
    """Deduplicated Latin name variants; the first one is the primary name."""
    order = list(NAME_KEYS)
    name = a.get("name")
    if isinstance(name, str) and any(map(is_cyr, name)) and not any(map(is_lat, name)):
        order.remove("name")                          # a Cyrillic-only name ranks after the Latin variants
        order.insert(order.index("name:sr"), "name")
    seen, names = set(), []
    for k in order:
        v = a.get(k)
        if not isinstance(v, str):
            continue
        for part in re.split(r"\s*;\s*|\s+/\s+", v):  # 'Лисина / Lisina', 'Gradski Caffe;Ice Spot'
            if not part.strip():
                continue
            key = match_key(part)
            if key and key not in seen:
                seen.add(key)
                names.append(translit(tidy(part)))
    for n in list(names):                             # abbreviation expansions become extra aliases
        exp = expand_abbr(n)
        if exp and match_key(exp) not in seen:
            seen.add(match_key(exp))
            names.append(exp)
    return names

all_names = attrs.map(build_names)
n_raw = attrs.map(lambda a: sum(len(re.split(r"\s*;\s*|\s+/\s+", a[k])) for k in NAME_KEYS if isinstance(a.get(k), str))).sum()
print(f"Rows with a name: {(all_names.map(len) > 0).sum()} of {len(all_names)}")
print(f"Name strings before dedup: {n_raw}, after: {all_names.map(len).sum()}")
print(f"Rows with abbreviation expansions: {all_names.map(lambda ns: any(expand_abbr(n) for n in ns)).sum()}")
print("Examples:")
for i in all_names[all_names.map(len) >= 3].index[:5]:
    print("  ", all_names[i])

Rows with a name: 30982 of 44149
Name strings before dedup: 44016, after: 33184
Rows with abbreviation expansions: 254
Examples:
   ['Univerzitetski klinički centar Republike Srpske', 'Sveučilišni klinički centar Republike Srpske', 'University Clinical Centre of the Republic of Srpska']
   ['Spomen područje Jasenovac', 'Spomen područje logor Jasenovac', 'Jasenovac concentration and extermination camp']
   ['Međunarodni aerodrom Sarajevo', 'Međunarodna zračna luka Sarajevo', 'Sarajevo International Airport', 'J.P. Međunarodni aerodrom "SARAJEVO" d.o.o. Sarajevo', 'javno preduzeće Međunarodni aerodrom SARAJEVO Sarajevo']
   ['Bardača', 'Berski rezervat Bardača', 'Bardača Wetland', 'Barski rezervat Bardača']
   ['XV', 'XV Uskrsnuće', 'XV Resurrection']


### 2.4 Vocabularies

- `LABELS`: bilingual (English / Bosnian) labels for the ~200 most frequent categories (99.5% of rows); other values fall back to the English tag value.
- `DENOMINATION`: refines places of worship into mosque / Orthodox / Catholic church or chapel / synagogue.
- Value vocabularies for list-like keys (`cuisine`, `sport`, `clothes`, `memorial`, historic period, …).
- `FEATURES` and the payment / fuel / diet / vehicle-service groups: phrases for yes/enum values. `no` has no phrase on purpose.
- `FILTER_ALLOWED`: values kept in the structured `filters` column, including `no`. Values in the wrong key (opening hours in `delivery`, `km` in `fee`) are not in the allowed sets and are dropped.

In [15]:
# bilingual labels (English, Bosnian) for the ~200 most frequent categories (99.5% of rows); others fall back to English
LABELS = {
    "peak": ("peak", "vrh"), "cafe": ("cafe", "kafić"), "restaurant": ("restaurant", "restoran"),
    "spring": ("spring", "izvor"), "school": ("school", "škola"), "supermarket": ("supermarket", "supermarket"),
    "park": ("park", "park"), "convenience": ("convenience store", "prodavnica"), "fuel": ("petrol station", "benzinska pumpa"),
    "memorial": ("memorial", "spomen-obilježje"), "swimming_pool": ("swimming pool", "bazen"),
    "clothes": ("clothing store", "prodavnica odjeće"), "pharmacy": ("pharmacy", "apoteka"),
    "playground": ("playground", "dječije igralište"), "atm": ("ATM", "bankomat"), "bakery": ("bakery", "pekara"),
    "bank": ("bank", "banka"), "hotel": ("hotel", "hotel"), "hairdresser": ("hairdresser", "frizerski salon"),
    "bar": ("bar", "bar"), "fast_food": ("fast food", "brza hrana"), "viewpoint": ("viewpoint", "vidikovac"),
    "car_repair": ("car repair", "auto servis"), "car_wash": ("car wash", "autopraonica"), "kiosk": ("kiosk", "kiosk"),
    "post_office": ("post office", "pošta"), "attraction": ("tourist attraction", "turistička atrakcija"),
    "clinic": ("clinic", "ambulanta"), "cave_entrance": ("cave", "pećina"), "waterfall": ("waterfall", "vodopad"),
    "guest_house": ("guest house", "pansion"), "garden": ("garden", "vrt"), "apartment": ("apartment", "apartman"),
    "sports_centre": ("sports centre", "sportski centar"), "bookmaker": ("betting shop", "kladionica"),
    "police": ("police station", "policija"), "company": ("company office", "firma"),
    "community_centre": ("community centre", "dom kulture"), "monument": ("monument", "spomenik"),
    "government": ("government office", "javna institucija"), "taxi": ("taxi stand", "taksi stajalište"),
    "kindergarten": ("kindergarten", "vrtić"), "picnic_site": ("picnic site", "izletište"), "butcher": ("butcher", "mesnica"),
    "archaeological_site": ("archaeological site", "arheološki lokalitet"),
    "charging_station": ("EV charging station", "punionica za električna vozila"), "shoes": ("shoe store", "prodavnica obuće"),
    "motel": ("motel", "motel"), "furniture": ("furniture store", "namještaj"), "pub": ("pub", "pivnica"),
    "dentist": ("dentist", "stomatolog"), "camp_site": ("campsite", "kamp"), "mall": ("shopping mall", "tržni centar"),
    "townhall": ("town hall", "općina"), "florist": ("florist", "cvjećara"), "car": ("car dealer", "auto salon"),
    "stadium": ("stadium", "stadion"), "alpine_hut": ("mountain hut", "planinarski dom"), "jewelry": ("jewelry store", "zlatara"),
    "marketplace": ("marketplace", "pijaca"), "hospital": ("hospital", "bolnica"), "chalet": ("chalet", "brvnara"),
    "museum": ("museum", "muzej"), "sports": ("sports store", "sportska oprema"), "car_parts": ("car parts", "auto dijelovi"),
    "electronics": ("electronics store", "elektronika"), "fire_station": ("fire station", "vatrogasna stanica"),
    "bus_station": ("bus station", "autobuska stanica"), "mobile_phone": ("mobile phone store", "mobiteli"),
    "insurance": ("insurance", "osiguranje"), "hostel": ("hostel", "hostel"), "veterinary": ("veterinarian", "veterinar"),
    "books": ("bookstore", "knjižara"), "nightclub": ("nightclub", "noćni klub"), "university": ("university", "fakultet"),
    "computer": ("computer store", "informatička oprema"), "telecommunication": ("telecom store", "telekom"),
    "doityourself": ("DIY store", "građevinski materijal"), "chemist": ("drugstore", "drogerija"),
    "castle": ("castle", "tvrđava"), "library": ("library", "biblioteka"), "beauty": ("beauty salon", "kozmetički salon"),
    "hardware": ("hardware store", "željezara"), "fitness_centre": ("gym", "teretana"),
    "social_facility": ("social facility", "socijalna ustanova"), "optician": ("optician", "optika"),
    "gift": ("gift shop", "pokloni"), "doctors": ("doctor's office", "ordinacija"), "casino": ("casino", "kazino"),
    "cosmetics": ("cosmetics store", "kozmetika"), "courthouse": ("courthouse", "sud"),
    "bureau_de_change": ("currency exchange", "mjenjačnica"), "travel_agency": ("travel agency", "turistička agencija"),
    "cinema": ("cinema", "kino"), "water_park": ("water park", "akva park"), "theatre": ("theatre", "pozorište"),
    "toys": ("toy store", "igračke"), "ticket": ("ticket office", "prodaja karata"), "greengrocer": ("greengrocer", "voće i povrće"),
    "houseware": ("houseware store", "kućne potrepštine"), "wilderness_hut": ("wilderness hut", "planinarsko sklonište"),
    "confectionery": ("confectionery", "slatkiši"), "bicycle": ("bicycle shop", "bicikli"), "trade": ("trade supplier", "veleprodaja"),
    "garden_centre": ("garden centre", "vrtni centar"), "car_rental": ("car rental", "rent-a-car"),
    "copyshop": ("copy shop", "fotokopirnica"), "ngo": ("NGO", "nevladina organizacija"), "lawyer": ("lawyer", "advokat"),
    "department_store": ("department store", "robna kuća"), "variety_store": ("variety store", "mješovita roba"),
    "money_lender": ("money lender", "mikrokreditna organizacija"), "tyres": ("tyre shop", "vulkanizer"),
    "perfumery": ("perfumery", "parfimerija"), "pet": ("pet shop", "pet shop"), "pastry": ("pastry shop", "slastičarna"),
    "lottery": ("lottery", "lutrija"), "agrarian": ("agricultural supplies", "poljoprivredna apoteka"),
    "hifi": ("hi-fi store", "audio oprema"), "appliance": ("home appliances", "bijela tehnika"), "beverages": ("beverage store", "pića"),
    "laboratory": ("medical laboratory", "laboratorija"), "photo": ("photo shop", "foto radnja"), "boutique": ("boutique", "butik"),
    "alcohol": ("liquor store", "alkoholna pića"), "seafood": ("fish shop", "ribarnica"),
    "estate_agent": ("real estate agency", "agencija za nekretnine"), "carpenter": ("carpenter", "stolar"),
    "ice_cream": ("ice cream", "sladoled"), "electrical": ("electrical supplies", "elektro materijal"),
    "laundry": ("laundry", "perionica"), "events_venue": ("event venue", "sala za događaje"),
    "bathroom_furnishing": ("bathroom furnishing", "sanitarije"), "wholesale": ("wholesale", "veleprodaja"),
    "gallery": ("art gallery", "galerija"), "newsagent": ("newsagent", "trafika"), "watches": ("watch store", "satovi"),
    "second_hand": ("second-hand shop", "polovna roba"), "aerodrome": ("airport", "aerodrom"),
    "accountant": ("accountant", "knjigovodstvo"), "health_food": ("health food", "zdrava hrana"),
    "photographer": ("photographer", "fotograf"), "shelter": ("shelter", "sklonište"), "college": ("college", "visoka škola"),
    "stationery": ("stationery", "papirnica"), "nature_reserve": ("nature reserve", "zaštićeno područje"),
    "artwork": ("artwork", "umjetničko djelo"), "employment_agency": ("employment agency", "zavod za zapošljavanje"),
    "paint": ("paint store", "boje i lakovi"), "shoemaker": ("shoemaker", "obućar"),
    "nutrition_supplements": ("supplements", "suplementi"), "e-cigarette": ("e-cigarette shop", "elektronske cigarete"),
    "carpet": ("carpet store", "tepisi"), "architect": ("architect", "arhitekt"),
    "fashion_accessories": ("fashion accessories", "modni dodaci"), "art": ("art shop", "umjetnine"),
    "arts_centre": ("arts centre", "kulturni centar"), "dry_cleaning": ("dry cleaning", "hemijska čistionica"),
    "tailor": ("tailor", "krojač"), "electrician": ("electrician", "električar"), "farm": ("farm shop", "farma"),
    "locksmith": ("locksmith", "bravar"), "lighting": ("lighting store", "rasvjeta"), "herbalist": ("herbalist", "biljna apoteka"),
    "shoe_repair": ("shoe repair", "popravka obuće"), "interior_decoration": ("interior decoration", "dekoracija"),
    "fabric": ("fabric store", "tekstil"), "dog_park": ("dog park", "park za pse"), "general": ("general store", "mješovita roba"),
    "bag": ("bag store", "torbe"), "fort": ("fort", "utvrda"), "music": ("music store", "muzička oprema"),
    "baby_goods": ("baby goods", "oprema za bebe"), "theme_park": ("theme park", "zabavni park"),
    "medical_supply": ("medical supplies", "medicinska oprema"), "bed": ("bed store", "madraci"), "doors": ("doors", "vrata"),
    "weapons": ("weapons store", "oružje"), "drinking_water": ("drinking water", "česma"), "bowling_alley": ("bowling", "kuglana"),
    "motorcycle": ("motorcycle shop", "motocikli"), "ice_rink": ("ice rink", "klizalište"),
    "caravan_site": ("caravan site", "kamp za kampere"), "tattoo": ("tattoo studio", "tattoo studio"),
    "religion": ("religious shop", "religijski artikli"), "funeral_directors": ("funeral home", "pogrebno preduzeće"),
    "coffee": ("coffee roaster", "pržionica kafe"), "radiotechnics": ("electronic components", "radio-tehnika"),
    "physiotherapist": ("physiotherapist", "fizioterapija"), "golf_course": ("golf course", "golf teren"),
    "printing": ("printing", "štamparija"), "food_court": ("food court", "food court"),
    "hairdresser_supply": ("hairdresser supplies", "frizerska oprema"), "zoo": ("zoo", "zoološki vrt"),
    # key-specific values
    "railway=station": ("railway station", "željeznička stanica"), "railway=halt": ("railway halt", "željeznička stanica"),
    "public_transport=station": ("transit station", "stanica"), "aeroway=terminal": ("airport terminal", "aerodromski terminal"),
    # generic "<key>=yes" rows
    "shop=yes": ("shop", "prodavnica"), "historic=yes": ("historic site", "historijski lokalitet"),
    "office=yes": ("office", "ured"), "tourism=yes": ("tourist site", "turističko mjesto"),
}

DENOMINATION = {"serbian_orthodox": ("Serbian Orthodox", "srpska pravoslavna"), "orthodox": ("Orthodox", "pravoslavna"),
                "roman_catholic": ("Catholic", "katolička"), "catholic": ("Catholic", "katolička"),
                "ukrainian_greek_catholic": ("Greek Catholic", "grkokatolička")}

# value vocabularies for list-like descriptive keys: value -> (English, Bosnian)
CUISINE = {"regional": ("regional", "domaća"), "local": ("local", "lokalna"), "bosnian": ("Bosnian", "bosanska"),
           "balkan": ("Balkan", "balkanska"), "pizza": ("pizza", "pizza"), "coffee_shop": ("coffee", "kafa"),
           "kebab": ("kebab", "kebab"), "burger": ("burger", "burger"), "grill": ("grill", "roštilj"),
           "barbecue": ("barbecue", "roštilj"), "italian": ("Italian", "italijanska"), "international": ("international", "internacionalna"),
           "fish": ("fish", "riba"), "seafood": ("seafood", "morski plodovi"), "sandwich": ("sandwiches", "sendviči"),
           "ice_cream": ("ice cream", "sladoled"), "cake": ("cakes", "kolači"), "chicken": ("chicken", "piletina"),
           "cevapi": ("ćevapi", "ćevapi"), "burek": ("burek", "burek"), "pita": ("pita", "pita"), "pasta": ("pasta", "tjestenina"),
           "steak_house": ("steak house", "odrezak"), "chinese": ("Chinese", "kineska"), "turkish": ("Turkish", "turska"),
           "mediterranean": ("Mediterranean", "mediteranska"), "breakfast": ("breakfast", "doručak"), "crepe": ("crêpes", "palačinke")}
SPORT = {"soccer": ("football", "fudbal"), "swimming": ("swimming", "plivanje"), "tennis": ("tennis", "tenis"),
         "basketball": ("basketball", "košarka"), "handball": ("handball", "rukomet"), "volleyball": ("volleyball", "odbojka"),
         "fitness": ("fitness", "fitnes"), "multi": ("multi-sport", "više sportova"), "ice_skating": ("ice skating", "klizanje"),
         "skiing": ("skiing", "skijanje"), "climbing": ("climbing", "penjanje"), "shooting": ("shooting", "streljaštvo"),
         "water_polo": ("water polo", "vaterpolo"), "running": ("running", "trčanje"), "bowling": ("bowling", "kuglanje"),
         "martial_arts": ("martial arts", "borilačke vještine"), "boxing": ("boxing", "boks"), "karate": ("karate", "karate")}
CLOTHES = {"women": ("women", "ženska"), "men": ("men", "muška"), "children": ("children", "dječija"),
           "underwear": ("underwear", "donji veš"), "sports": ("sportswear", "sportska"), "wedding": ("wedding", "vjenčanice"),
           "suits": ("suits", "odijela")}
MEMORIAL = {"war_memorial": ("war memorial", "spomenik palim borcima"), "plaque": ("memorial plaque", "spomen-ploča"),
            "bust": ("bust", "bista"), "statue": ("statue", "statua"), "stone": ("memorial stone", "spomen-kamen"),
            "sculpture": ("sculpture", "skulptura"), "obelisk": ("obelisk", "obelisk"), "stele": ("stele", "stela"),
            "cross": ("memorial cross", "spomen-krst"), "stolperstein": ("stolperstein", "kamen spoticanja")}
PERIOD = {"medieval": ("medieval", "srednjovjekovni"), "middle-ages": ("medieval", "srednjovjekovni"),
          "ottoman": ("Ottoman", "osmanski"), "ancient_roman": ("Roman", "rimski"), "roman": ("Roman", "rimski"),
          "ancient_greek": ("ancient Greek", "antički grčki"), "illyrian": ("Illyrian", "ilirski"),
          "prehistoric": ("prehistoric", "prahistorijski"), "austro-hungarian": ("Austro-Hungarian", "austrougarski"),
          "modern": ("modern", "moderni"), "yugoslav": ("Yugoslav", "jugoslovenski")}
SOCIAL_FOR = {"senior": ("seniors", "starije osobe"), "child": ("children", "djeca"), "juvenile": ("youth", "mladi"),
              "disabled": ("persons with disabilities", "osobe s invaliditetom"), "homeless": ("homeless", "beskućnici"),
              "orphan": ("orphans", "djeca bez roditeljskog staranja")}
CASTLE = {"defensive": ("fortress", "utvrda"), "stately": ("manor", "dvorac"), "castrum": ("Roman fort", "rimski kastrum"),
          "fortress": ("fortress", "utvrda"), "palace": ("palace", "palata")}
MUSEUM = {"history": ("history", "historijski"), "art": ("art", "umjetnički"), "military": ("military", "vojni"),
          "local": ("local history", "zavičajni"), "nature": ("natural history", "prirodnjački"), "war": ("war", "ratni")}
LIST_KEYS = {   # key -> (vocabulary, English prefix, Bosnian prefix); None prefix = items only
    "cuisine": (CUISINE, "cuisine", "kuhinja"), "sport": (SPORT, "sport", "sport"), "clothes": (CLOTHES, "clothes", "odjeća"),
    "memorial": (MEMORIAL, None, None), "historic:civilization": (PERIOD, "period", "period"),
    "historic:period": (PERIOD, "period", "period"), "castle_type": (CASTLE, None, None), "museum": (MUSEUM, "museum", "muzej"),
    "social_facility:for": (SOCIAL_FOR, "for", "za"), "healthcare:speciality": ({}, "speciality", "specijalnost"),
    "artwork_type": (MEMORIAL, None, None), "garden:type": ({}, "garden", "vrt"), "theatre:type": ({}, "theatre", "pozorište"),
}

# yes/no and enumerated features: key -> {normalized value: (English, Bosnian)}; "no" is never verbalized
FEATURES = {
    "wheelchair": {"yes": ("wheelchair accessible", "pristupačno za invalidska kolica"),
                   "limited": ("limited wheelchair access", "djelimično pristupačno za invalidska kolica")},
    "outdoor_seating": {"yes": ("outdoor seating", "bašta")},
    "indoor_seating": {"yes": ("indoor seating", "unutrašnja sala")},
    "air_conditioning": {"yes": ("air conditioning", "klima")},
    "smoking": {"yes": ("smoking allowed", "dozvoljeno pušenje"), "outside": ("smoking outside only", "pušenje samo vani"),
                "separated": ("separate smoking area", "odvojen prostor za pušače"), "dedicated": ("smoking venue", "prostor za pušače")},
    "takeaway": {"yes": ("takeaway", "hrana za ponijeti"), "only": ("takeaway only", "samo za ponijeti")},
    "delivery": {"yes": ("delivery", "dostava"), "only": ("delivery only", "samo dostava")},
    "drive_through": {"yes": ("drive-through", "drive-through")},
    "self_service": {"yes": ("self-service", "samoposluga"), "only": ("self-service only", "samo samoposluga"),
                     "partially": ("partial self-service", "djelimična samoposluga")},
    "toilets": {"yes": ("toilets", "toalet")},
    "toilets:wheelchair": {"yes": ("wheelchair accessible toilet", "toalet za osobe s invaliditetom")},
    "changing_table": {"yes": ("baby changing table", "sto za presvlačenje beba")},
    "reservation": {"yes": ("reservations", "rezervacije")},
    "dog": {"yes": ("dogs allowed", "dozvoljeni psi"), "leashed": ("dogs on leash allowed", "dozvoljeni psi na povocu")},
    "seasonal": {"yes": ("seasonal", "sezonski")},
    "fee": {"yes": ("entrance fee", "ulaz se plaća"), "donation": ("donation", "dobrovoljni prilog")},
    "drinking_water": {"yes": ("drinking water", "pitka voda")},
    "intermittent": {"yes": ("intermittent", "povremeni")},
    "emergency": {"yes": ("emergency service", "hitna služba")},
    "atm": {"yes": ("ATM", "bankomat")},
    "cash_in": {"yes": ("cash deposit", "uplata gotovine")},
    "dispensing": {"yes": ("dispenses prescriptions", "izdavanje lijekova na recept")},
    "compressed_air": {"yes": ("compressed air", "kompresor za gume")},
    "automated": {"yes": ("automated", "automatizovano")},
    "tents": {"yes": ("tents allowed", "šatori")},
    "caravans": {"yes": ("caravans allowed", "kamp prikolice")},
    "shower": {"yes": ("showers", "tuševi")},
    "covered": {"yes": ("covered", "natkriveno")},
    "indoor": {"yes": ("indoor", "zatvoreni prostor")},
    "second_hand": {"yes": ("second-hand goods", "polovna roba"), "only": ("second-hand only", "samo polovna roba")},
    "bus": {"yes": ("bus", "autobus")},
    "train": {"yes": ("train", "voz")},
}
VALUE_ALIASES = {"outdoor_seating": {"veranda": "yes", "patio": "yes"}, "wheelchair": {"designated": "yes"},
                 "shower": {"hot": "yes", "cold": "yes"}, "smoking": {"isolated": "separated"},
                 "reservation": {"recommended": "yes", "required": "yes"}, "internet_access": {"wifi": "wlan"},
                 "internet_access:fee": {"no @ customers": "customers"}}

PAYMENT = {"cash": ("cash", "gotovina"), "coins": ("coins", "kovanice"),
           **{k: ("cards", "kartice") for k in ["credit_cards", "debit_cards", "cards", "visa", "mastercard", "maestro",
                                                "visa_electron", "visa_debit", "american_express", "diners_club",
                                                "discover_card", "v_pay"]},
           **{k: ("contactless", "beskontaktno") for k in ["contactless", "google_pay", "apple_pay"]}}
FUEL = {"diesel": ("diesel", "dizel"), "octane_95": ("petrol 95", "benzin 95"), "octane_98": ("petrol 98", "benzin 98"),
        "octane_100": ("petrol 100", "benzin 100"), "lpg": ("LPG", "autoplin"), "cng": ("CNG", "CNG"), "adblue": ("AdBlue", "AdBlue")}
DIET = {"halal": ("halal", "halal"), "vegetarian": ("vegetarian", "vegetarijanska hrana"), "vegan": ("vegan", "veganska hrana"),
        "gluten_free": ("gluten-free", "bez glutena"), "kosher": ("kosher", "košer")}
VEHICLE_SERVICE = {"car_repair": ("car repair", "auto servis"), "tyres": ("tyres", "vulkanizer"),
                   "oil_change": ("oil change", "zamjena ulja"), "car_parts": ("car parts", "auto dijelovi"),
                   "body_repair": ("body repair", "limarija"), "electrical": ("auto electrics", "autoelektrika"),
                   "diagnostics": ("diagnostics", "dijagnostika"), "glass": ("car glass", "auto stakla"),
                   "painting": ("car painting", "lakiranje"), "air_conditioning": ("car A/C", "auto klima"),
                   "batteries": ("batteries", "akumulatori"), "new_car_sales": ("new cars", "nova vozila"),
                   "used_car_sales": ("used cars", "polovna vozila"), "wheel_alignment": ("wheel alignment", "trap")}

# values kept in the structured `filters` column (including "no")
FILTER_ALLOWED = {k: set(m) | {"no"} for k, m in FEATURES.items()}
FILTER_ALLOWED.update({"access": {"yes", "private", "customers", "permissive", "permit", "no"},
                       "internet_access": {"wlan", "yes", "terminal", "wired", "no"},
                       "internet_access:fee": {"yes", "no", "customers"}})
GROUP_PREFIXES = ("payment:", "fuel:", "diet:", "service:vehicle:")
print(f"Category labels: {len(LABELS)}, feature keys: {len(FEATURES)}")

Category labels: 202, feature keys: 32


### 2.5 Verbalization functions

- `category_pair`: category label, with the place-of-worship refinement.
- `features`: one phrase per positive feature; payment, fuel, diet and vehicle services are combined into one phrase per group (*payment: cash, cards / plaćanje: gotovina, kartice*); `internet_access` + `internet_access:fee=no` → *free Wi-Fi / besplatan WiFi*.
- `descriptors`: brand (max 5, skipped if equal to the name), operator (without legal form), list-like keys, heritage (*National Monument of BiH*, *UNESCO World Heritage Site*), elevation for peaks, hotel stars, diocese/parish.
- `free_texts`: `description`, `description:en`, `inscription` and `note`. A `note` with QA wording (*approximate*, *verify*, *resurvey*, …) is skipped. URLs are removed, each text is capped at 300 characters, and duplicates or texts equal to a name are skipped.
- `build_nlp_text`: `name. category labels. also known as: … descriptors. features. address: … free text`. A label part equal to the name is dropped (*Apoteka* → *pharmacy*, not *pharmacy / apoteka*).
- `filters_of`, `contact_of`: the structured columns. Phones are normalized to E.164 (`+38733…`).

In [16]:
def bi(en, bs=None):
    """'pharmacy / apoteka'; a single phrase when both languages are the same."""
    return en if not bs or bs.casefold() == en.casefold() else f"{en} / {bs}"

def category_of(a):
    """(key, value) of the primary category; generic '<key>=yes' only when nothing more specific exists."""
    for k in CAT_KEYS:
        v = a.get(k)
        if isinstance(v, str) and v != "yes":
            return k, v.split(";")[0].strip()
    for k in CAT_KEYS:
        if a.get(k) == "yes":
            return k, "yes"
    return None, None

def category_pair(key, value, a):
    """(English, Bosnian) label of a category, refined by religion for places of worship."""
    if value == "place_of_worship":
        rel, den = a.get("religion"), a.get("denomination")
        if rel == "muslim":
            return ("mosque", "džamija")
        if rel == "jewish":
            return ("synagogue", "sinagoga")
        if rel == "christian":
            noun = ("chapel", "kapela") if a.get("building") == "chapel" else ("church", "crkva")
            adj = DENOMINATION.get(den)
            return (f"{adj[0]} {noun[0]}", f"{adj[1]} {noun[1]}") if adj else noun
        return ("place of worship", "bogomolja")
    return LABELS.get(f"{key}={value}") or LABELS.get(value) or (value.replace("_", " "), None)

def norm_feature(key, v):
    if not isinstance(v, str):
        return None
    v = v.strip().lower()
    return VALUE_ALIASES.get(key, {}).get(v, v)

def list_phrase(raw, vocab, en_lbl, bs_lbl, cap=6):
    items = [v.strip() for v in re.split(r"[;,]", raw) if v.strip() and v.strip() != "yes"][:cap]
    if not items:
        return None
    pairs = [vocab.get(v, (v.replace("_", " "),) * 2) for v in items]
    en = ", ".join(dict.fromkeys(p[0] for p in pairs))
    bs = ", ".join(dict.fromkeys(p[1] for p in pairs))
    return bi(f"{en_lbl}: {en}", f"{bs_lbl}: {bs}") if en_lbl else bi(en, bs)

def group_phrase(a, prefix, vocab, en_lbl, bs_lbl):
    found = {}
    for k, v in a.items():
        if k.startswith(prefix) and norm_feature(k, v) in ("yes", "only"):
            pair = vocab.get(k[len(prefix):])
            if pair:
                only = norm_feature(k, v) == "only"
                found[pair] = found.get(pair, False) or only
    if not found:
        return None
    en = ", ".join(f"{p[0]} only" if only else p[0] for p, only in found.items())
    bs = ", ".join(f"samo {p[1]}" if only else p[1] for p, only in found.items())
    return bi(f"{en_lbl}: {en}", f"{bs_lbl}: {bs}")

def features(a, skip_en):
    out = []
    for key, mapping in FEATURES.items():
        v = norm_feature(key, a.get(key))
        if v in mapping and mapping[v][0] not in skip_en:
            out.append(bi(*mapping[v]))
    net, fee = norm_feature("internet_access", a.get("internet_access")), norm_feature("internet_access:fee", a.get("internet_access:fee"))
    if net in ("wlan", "yes", "terminal", "wired"):
        what = ("Wi-Fi", "WiFi") if net == "wlan" else ("internet access", "internet")
        if fee == "no":
            out.append(bi(f"free {what[0]}", f"besplatan {what[1]}"))
        elif fee == "customers":
            out.append(bi(f"{what[0]} for customers", f"{what[1]} za goste"))
        else:
            out.append(bi(*what))
    out += [p for p in (group_phrase(a, "payment:", PAYMENT, "payment", "plaćanje"),
                        group_phrase(a, "fuel:", FUEL, "fuel", "gorivo"),
                        group_phrase(a, "diet:", DIET, "diet", "ishrana"),
                        group_phrase(a, "service:vehicle:", VEHICLE_SERVICE, "services", "usluge")) if p]
    if "EV charging station" not in skip_en and any(k.startswith("socket:") and str(v).lower() not in ("no", "0") for k, v in a.items()):
        out.append(bi("EV charging", "punjenje električnih vozila"))
    return list(dict.fromkeys(out))

def split_list(v, cap=5):
    return [p.strip() for p in re.split(r"\s*;\s*", v) if p.strip()][:cap] if isinstance(v, str) else []

def plural_stars(n):
    return f"{n} zvjezdica" if n == 1 or n >= 5 else f"{n} zvjezdice"

def descriptors(a, cat_val, name_keys):
    out = []
    brands = [b for b in (text_form(x) for x in split_list(a.get("brand"))) if match_key(b) not in name_keys]
    if brands:
        out.append("brand: " + ", ".join(brands))
    skip = name_keys | {match_key(b) for b in brands}
    ops = [o for o in (text_form(x) for x in split_list(a.get("operator"))) if match_key(o) not in skip]
    if ops:
        out.append("operator: " + ", ".join(ops))
    for key, (vocab, en_lbl, bs_lbl) in LIST_KEYS.items():
        if isinstance(a.get(key), str) and (p := list_phrase(a[key], vocab, en_lbl, bs_lbl)):
            out.append(p)
    op = str(a.get("heritage:operator", "")).lower()
    if "whc" in op:
        out.append(bi("UNESCO World Heritage Site", "UNESCO svjetska baština"))
    if "kons" in op:
        out.append(bi("National Monument of Bosnia and Herzegovina", "nacionalni spomenik BiH"))
    elif a.get("heritage") and "whc" not in op:
        out.append(bi("protected heritage", "zaštićeno kulturno naslijeđe"))
    ele = a.get("ele")
    if cat_val in ("peak", "hill", "saddle", "ridge", "volcano") and isinstance(ele, str) and re.fullmatch(r"\d+(\.\d+)?", ele):
        m = round(float(ele))
        out.append(bi(f"elevation {m} m", f"nadmorska visina {m} m"))
    stars = a.get("stars")
    if isinstance(stars, str) and stars.isdigit() and 1 <= int(stars) <= 5:
        out.append(bi(f"{stars}-star", plural_stars(int(stars))))
    if cat_val != "place_of_worship" and a.get("religion") in ("muslim", "christian", "jewish"):
        out.append(bi(*{"muslim": ("Muslim", "islamski"), "christian": ("Christian", "kršćanski"),
                        "jewish": ("Jewish", "jevrejski")}[a["religion"]]))
    for key in ("diocese", "parish"):
        if isinstance(a.get(key), str):
            out.append(text_form(a[key]))
    return out

def address_text(a):
    street = " ".join(translit(tidy(a[k])) for k in ("addr:street", "addr:housenumber") if isinstance(a.get(k), str))
    bits = [street] + [translit(tidy(a[k])) for k in ("addr:place", "addr:city") if isinstance(a.get(k), str)]
    return ", ".join(dict.fromkeys(b for b in bits if b))

QA_RE = re.compile(r"geocod|verif|approximate|incomplete|resurvey|fixme|survey", re.I)
URL_RE = re.compile(r"https?://\S+|www\.\S+")

def free_texts(a, name_keys, cap=300):
    out, seen = [], set(name_keys)
    for k in ("description", "description:en", "inscription", "note"):
        v = a.get(k)
        if not isinstance(v, str) or (k == "note" and QA_RE.search(v)):
            continue
        t = re.sub(r"\s+", " ", URL_RE.sub("", translit(tidy(v)))).strip(" ;,.-")
        if len(t) > cap:
            t = t[:cap].rsplit(" ", 1)[0] + " …"
        if t and match_key(t) not in seen:
            seen.add(match_key(t))
            out.append(t)
    return out

def build_nlp_text(a, names, cat_key, cat_val):
    name_keys = {match_key(n) for n in names}
    parts = [text_form(names[0])] if names else []
    labels, skip_en = [], set()
    cats = [(cat_key, cat_val)] if cat_val else []
    cats += [(k, a[k].split(";")[0].strip()) for k in CAT_KEYS
             if k != cat_key and isinstance(a.get(k), str) and a[k] != "yes"]
    for k, v in cats:
        en, bs = category_pair(k, v, a)
        skip_en.add(en)
        kept = [p for p in (en, bs) if p and match_key(p) not in name_keys]   # 'Apoteka' -> drop 'apoteka'
        if kept:
            labels.append(bi(*kept) if len(kept) == 2 else kept[0])
    if labels:
        parts.append(", ".join(dict.fromkeys(labels)))
    if len(names) > 1:
        parts.append("also known as: " + "; ".join(text_form(n) for n in names[1:]))
    parts += descriptors(a, cat_val, name_keys)
    if feats := features(a, skip_en):
        parts.append(", ".join(feats))
    if addr := address_text(a):
        parts.append(f"address: {addr}")
    parts += free_texts(a, name_keys)
    return ". ".join(p.strip().rstrip(".") for p in parts if p and p.strip())

def filters_of(a):
    f = {}
    for k, v in a.items():
        nv = norm_feature(k, v)
        allowed = FILTER_ALLOWED.get(k) or ({"yes", "no", "only"} if k.startswith(GROUP_PREFIXES) else None)
        if allowed and nv in allowed:
            f[k] = nv
    return f

def e164(p):
    d = re.sub(r"[^\d+]", "", p)
    if d.startswith("+"):
        d = d[1:]
    elif d.startswith("00"):
        d = d[2:]
    elif d.startswith("0"):
        d = "387" + d[1:]
    elif len(d) in (8, 9):
        d = "387" + d
    return "+" + d if 10 <= len(d) <= 13 and d.isdigit() else None

def contact_of(a):
    phones = []
    for k in ("phone", "contact:phone", "mobile", "contact:mobile", "phone_1"):
        for p in re.split(r"[;,]", a[k]) if isinstance(a.get(k), str) else []:
            if (n := e164(p)) and n not in phones:
                phones.append(n)
    web = next((a[k] for k in ("website", "contact:website", "url") if isinstance(a.get(k), str)), None)
    if web:
        web = web.split(";")[0].strip()
        web = web if re.match(r"https?://", web, re.I) else "https://" + web
    email = next((a[k] for k in ("email", "contact:email") if isinstance(a.get(k), str)), None)
    out = {"phones": phones, "website": web, "email": email.split(";")[0].strip().lower() if email else None,
           "facebook": a.get("contact:facebook") or a.get("facebook"), "instagram": a.get("contact:instagram")}
    return {k: v for k, v in out.items() if v}

### 2.6 Build `gdf_clean`

`loc_approx` marks rows whose `note`/`source` says the location comes from a geocoder.

In [17]:
records = []
for idx, a in attrs.items():
    names = all_names[idx]
    cat_key, cat_val = category_of(a)
    en, bs = category_pair(cat_key, cat_val, a) if cat_val else (None, None)
    contact, flt = contact_of(a), filters_of(a)
    postcode = a.get("addr:postcode")
    city = translit(tidy(a["addr:city"])) if isinstance(a.get("addr:city"), str) else None
    records.append({
        "primary_name": names[0] if names else None,
        "aliases": names[1:],
        "primary_category": None if cat_val is None else (cat_key if cat_val == "yes" else cat_val),
        "category_key": cat_key,
        "category_label": bi(en, bs) if en else None,
        "address": ", ".join(b for b in [address_text({k: v for k, v in a.items() if k != "addr:city"}),
                                          " ".join(x for x in [postcode, city] if isinstance(x, str))] if b) or None,
        "city": city,
        "postcode": postcode if isinstance(postcode, str) else None,
        "opening_hours": a.get("opening_hours"),
        "contact": json.dumps(contact, ensure_ascii=False) if contact else None,
        "filters": json.dumps(flt, ensure_ascii=False) if flt else None,
        "wikidata": parse_tags(gdf.at[idx, "tags"]).get("wikidata"),
        "loc_approx": any(isinstance(a.get(k), str) and re.search(r"geocod|approximate", a[k], re.I) for k in ("note", "source")),
        "nlp_text": build_nlp_text(a, names, cat_key, cat_val),
    })

gdf_clean = gpd.GeoDataFrame(
    pd.DataFrame(records, index=gdf.index).assign(
        id=gdf["id"], osm_type=gdf["osm_type"], lat=gdf["lat"], lon=gdf["lon"], merged_ids=gdf["merged_ids"]),
    geometry=gdf.geometry, crs=gdf.crs)
gdf_clean = gdf_clean[["id", "osm_type", "primary_name", "aliases", "primary_category", "category_key", "category_label",
                       "lat", "lon", "address", "city", "postcode", "opening_hours", "contact", "filters",
                       "wikidata", "loc_approx", "merged_ids", "nlp_text", "geometry"]]

n_words = gdf_clean["nlp_text"].str.split().str.len()
print(f"gdf_clean: {gdf_clean.shape[0]} rows x {gdf_clean.shape[1]} columns")
print(f"  with primary_name: {gdf_clean['primary_name'].notna().sum()}, with primary_category: {gdf_clean['primary_category'].notna().sum()}")
print(f"  with contact: {gdf_clean['contact'].notna().sum()}, with filters: {gdf_clean['filters'].notna().sum()}, "
      f"with opening_hours: {gdf_clean['opening_hours'].notna().sum()}, loc_approx: {gdf_clean['loc_approx'].sum()}")
print(f"  missing lat/lon: {(gdf_clean['lat'].isna() | gdf_clean['lon'].isna()).sum()}, empty nlp_text: {(gdf_clean['nlp_text'] == '').sum()}")
print(f"  nlp_text words: median {n_words.median():.0f}, 90% {n_words.quantile(.9):.0f}, 99% {n_words.quantile(.99):.0f}, max {n_words.max()}")

gdf_clean: 44149 rows x 20 columns
  with primary_name: 30982, with primary_category: 44149
  with contact: 5183, with filters: 6132, with opening_hours: 4164, loc_approx: 9
  missing lat/lon: 0, empty nlp_text: 0
  nlp_text words: median 7, 90% 15, 99% 32, max 98


### 2.7 Sample rows

Five different kinds of rows: a café with many features, an Orthodox church with a Cyrillic name, a peak, a school with an expanded abbreviation, and a national monument mapped as a polygon (filled coordinates).

In [18]:
def pick(mask):
    hits = gdf_clean.index[mask & ~gdf_clean.index.isin(picked)]
    if len(hits):
        picked.append(hits[0])

picked = []
has = lambda col: gdf_clean[col].notna()
n_yes = gdf_clean["filters"].map(lambda f: sum(v in ("yes", "only", "limited") for v in json.loads(f).values()) if isinstance(f, str) else 0)
pick(gdf_clean["primary_category"].isin(["cafe", "restaurant"]) & n_yes.ge(5) & has("address"))
pick((gdf_clean["primary_category"] == "place_of_worship") & gdf["name"].fillna("").map(lambda s: any(map(is_cyr, s))))
pick((gdf_clean["primary_category"] == "peak") & gdf_clean["aliases"].map(len).gt(0))
pick(gdf_clean["aliases"].map(lambda al: any("osnovna škola" in x for x in al)))
pick(gdf_clean["nlp_text"].str.contains("National Monument") & gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"]))

for i in picked:
    r = gdf_clean.loc[i]
    print("=" * 100)
    for col in ["id", "osm_type", "primary_name", "aliases", "primary_category", "category_label", "lat", "lon",
                "address", "opening_hours", "contact", "filters", "wikidata"]:
        if r[col] is not None and not (isinstance(r[col], float) and np.isnan(r[col])) and not (isinstance(r[col], list) and not r[col]):
            print(f"{col:17s}: {r[col]}")
    print(f"{'nlp_text':17s}: {r['nlp_text']}")

id               : 9420678500
osm_type         : node
primary_name     : Box
primary_category : cafe
category_label   : cafe / kafić
lat              : 44.8069293
lon              : 17.2077903
address          : Branka Popovića 41G, 78000 Banja Luka
opening_hours    : Mo-Su 07:00-23:00
filters          : {"air_conditioning": "yes", "changing_table": "no", "diet:gluten_free": "no", "diet:halal": "no", "diet:kosher": "no", "diet:vegetarian": "no", "indoor_seating": "yes", "internet_access:fee": "no", "payment:credit_cards": "no", "payment:debit_cards": "no", "smoking": "yes", "internet_access": "wlan", "outdoor_seating": "yes", "wheelchair": "yes"}
nlp_text         : Box. cafe / kafić. wheelchair accessible / pristupačno za invalidska kolica, outdoor seating / bašta, indoor seating / unutrašnja sala, air conditioning / klima, smoking allowed / dozvoljeno pušenje, free Wi-Fi / besplatan WiFi. address: Branka Popovića 41G, Banja Luka
id               : 194463895
osm_type         : way
prim

## 3. Save the cleaned dataset

`gdf_clean` is written to `data/processed/pois_clean.parquet` (GeoParquet, EPSG:4326). `aliases` and `merged_ids` are stored as list columns; `contact` and `filters` as JSON strings (`json.loads` to read them back). The file is read back once to check that rows, columns, CRS and list columns survived the round trip.

In [19]:
from pathlib import Path

OUT_PATH = Path("../data/processed/pois_clean.parquet")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

gdf_clean.to_parquet(OUT_PATH, index=False)

# round-trip check
check = gpd.read_parquet(OUT_PATH)
assert len(check) == len(gdf_clean) and list(check.columns) == list(gdf_clean.columns)
assert check.crs.to_epsg() == 4326
assert check["id"].is_unique and check[["lat", "lon"]].notna().all().all()
print(f"Saved {len(check)} rows x {check.shape[1]} columns to {OUT_PATH.resolve()} ({OUT_PATH.stat().st_size / 1e6:.1f} MB)")
print(f"aliases type after reload: {type(check['aliases'].iloc[0]).__name__}, "
      f"merged_ids type: {type(check['merged_ids'].iloc[0]).__name__}")

Saved 44149 rows x 20 columns to C:\Users\Korisnik\GeoIR\data\processed\pois_clean.parquet (5.9 MB)
aliases type after reload: ndarray, merged_ids type: ndarray


In [21]:
gdf_clean.head(10)

,id,osm_type,primary_name,aliases,primary_category,category_key,category_label,lat,lon,address,city,postcode,opening_hours,contact,filters,wikidata,loc_approx,merged_ids,nlp_text,geometry
376,606256941,node,MUP Srpske,[],police,amenity,police station / policija,45.047300,16.380927,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,[],MUP Srpske. police station / policija,POINT (16.38093 45.0473)
35372,422036762,way,Park prirode Lonjsko polje,[],nature_reserve,leisure,nature reserve / zaštićeno područje,45.360888,16.820888,NaN,NaN,NaN,NaN,"{""phones"": [""+38544672080""], ""website"": ""https...",NaN,Q1933800,False,[],Park prirode Lonjsko polje. nature reserve / z...,"POLYGON ((16.81963 45.44187, 16.81816 45.44086..."
31255,109384440,way,Univerzitet u Banjoj Luci,[University of Banja Luka],university,amenity,university / fakultet,44.775151,17.211454,"Bulevar vojvode Petra Bojovića 1a, 78000",NaN,78000,NaN,"{""website"": ""https://unibl.org/"", ""facebook"": ...",NaN,Q1587312,False,[],Univerzitet u Banjoj Luci. university / fakult...,"POLYGON ((17.21269 44.77761, 17.21262 44.7776,..."
35552,444333266,way,Stari most,[Old bridge],attraction,tourism,tourist attraction / turistička atrakcija,43.337264,17.815037,NaN,NaN,NaN,NaN,NaN,"{""toilets:wheelchair"": ""no"", ""wheelchair"": ""no""}",Q188528,False,[],Stari most. tourist attraction / turistička at...,"POLYGON ((17.81487 43.33725, 17.8152 43.33725,..."
32282,184429388,way,Gradski park,[],park,leisure,park,44.812665,15.870808,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,[],Gradski park. park,"POLYGON ((15.86976 44.8134, 15.86882 44.8133, ..."
33225,220225359,way,Univerzitetski klinički centar Republike Srpske,"[Sveučilišni klinički centar Republike Srpske,...",hospital,amenity,hospital / bolnica,44.783584,17.178481,"Dvanaest beba bb, 78000 Banja Luka",Banja Luka,78000,NaN,"{""phones"": [""+38751342100""], ""website"": ""https...","{""emergency"": ""yes""}",Q24702916,False,[],Univerzitetski klinički centar Republike Srpsk...,"POLYGON ((17.18103 44.78484, 17.18085 44.78486..."
35704,474428386,way,Katedrala Srca Isusova,[Sacred Heart Cathedral],place_of_worship,amenity,Catholic church / katolička crkva,43.859430,18.425363,"Trg fra Grge Martića, 71000 Sarajevo",Sarajevo,71000,NaN,"{""website"": ""https://katedrala-sarajevo.com/""}","{""wheelchair"": ""no""}",Q1267112,False,[],Katedrala Srca Isusova. Catholic church / kato...,"POLYGON ((18.42523 43.85954, 18.4252 43.85953,..."
31463,145986106,way,Gimnazija,[],school,amenity,school / škola,44.769998,17.183442,"Zmaj Jovina 13, 78000 Banja Luka",Banja Luka,78000,NaN,"{""phones"": [""+38751213259""], ""website"": ""https...",NaN,NaN,False,[],Gimnazija. school / škola. operator: JU Gimnaz...,"POLYGON ((17.18285 44.76948, 17.18361 44.76911..."
31159,79494788,way,Spomen područje Jasenovac,"[Spomen područje logor Jasenovac, Jasenovac co...",attraction,tourism,tourist attraction / turistička atrakcija,45.279551,16.929937,NaN,NaN,NaN,NaN,"{""website"": ""https://www.jusp-jasenovac.hr/Def...",NaN,Q155032,False,[],Spomen područje Jasenovac. tourist attraction ...,"POLYGON ((16.92018 45.27881, 16.92176 45.27785..."
35695,472086702,way,Međunarodni aerodrom Sarajevo,"[Međunarodna zračna luka Sarajevo, Sarajevo In...",aerodrome,aeroway,airport / aerodrom,43.825362,18.333147,"Kurta Schorka 36, 71210 Sarajevo",Sarajevo,71210,NaN,"{""phones"": [""+38733289100""], ""website"": ""https...",NaN,Q740392,False,[],Međunarodni aerodrom Sarajevo. airport / aerod...,"POLYGON ((18.33309 43.82738, 18.33306 43.82736..."
